# 02 Preprocessing: add synthetic prices

ABO listings have no price field. This notebook adds **simulated (synthetic) prices** so price filtering (e.g. 'under $50') can be demonstrated. They are not real prices.

Prerequisites: `01_data_exploration.ipynb` saved `listings_en_raw.parquet` to Drive, and `src/preprocess/synthetic_prices.py` is committed to the repo.

In [6]:
import os
from google.colab import drive
drive.mount('/content/drive')
REPO = '/content/shoptalk'
if not os.path.exists(REPO):
    try:
        from google.colab import userdata
        token = userdata.get('GITHUB_TOKEN')
        url = f'https://{token}@github.com/anbuaru/shoptalk.git'
    except Exception:
        url = 'https://github.com/anbuaru/shoptalk.git'
    !git clone {url} {REPO}
else:
    !git -C {REPO} pull

%cd /content/shoptalk
!pip install -q -r requirements.txt

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
remote: Enumerating objects: 7, done.
remote: Counting objects: 100% (7/7), done.
remote: Compressing objects: 100% (4/4), done.
remote: Total 4 (delta 1), reused 0 (delta 0), pack-reused 0 (from 0)
Unpacking objects: 100% (4/4), 3.07 KiB | 3.07 MiB/s, done.
From https://github.com/anbuaru/shoptalk
   4930e3b..2584f06  main       -> origin/main
Updating 4930e3b..2584f06
Fast-forward
 preprocess/02_preprocessing.ipynb | 62 +++++++++++++++++++++++++++++++++------
 1 file changed, 53 insertions(+), 9 deletions(-)
/content/shoptalk
ERROR: Could not open requirements file: [Errno 2] No such file or directory: 'requirements.txt'


In [7]:
import sys
import os
import pandas as pd

sys.path.append(REPO)
assert os.path.exists(f'{REPO}/preprocess/synthetic_prices.py'), 'Commit preprocess/synthetic_prices.py to GitHub first'
from preprocess.synthetic_prices import synthetic_price

PROJECT_DIR = '/content/drive/MyDrive/IK classes/30_week 04Oct2026-LLM Ops - AI System Design- Sayan/ShopTalk'
df = pd.read_parquet(f'{PROJECT_DIR}/data/listings_en_raw.parquet')
print(df.shape)
df.head(3)

(147702, 10)


,item_id,name_en,brand_en,color_en,ptype,bullets_en,desc_en,keywords_en,main_image_id,country
0,B06X9STHNG,None,None,None,SHOES,,,,81iZlv3bjpL,NL
1,B07P8ML82R,None,None,None,HARDWARE,,,,619y9YG9cnL,MX
2,B07H9GMYXS,"AmazonBasics PETG 3D Printer Filament, 1.75mm,...",AmazonBasics,Translucent Yellow,MECHANICAL_COMPONENTS,3D printer filament with 1.75mm diameter + / -...,,"3d printer filament, petg printer filament, pe...",81NP7qh2L6L,AE


In [8]:
df['price'] = [synthetic_price(i, t) for i, t in zip(df['item_id'], df['ptype'])]
df['price_currency'] = 'USD'
df['price_is_synthetic'] = True
df[['item_id', 'ptype', 'price', 'price_is_synthetic']].head(10)

,item_id,ptype,price,price_is_synthetic
0,B06X9STHNG,SHOES,54.99,True
1,B07P8ML82R,HARDWARE,44.99,True
2,B07H9GMYXS,MECHANICAL_COMPONENTS,32.99,True
3,B07CTPR73M,SOFA,500.99,True
4,B01MTEI8M6,SHOES,38.99,True
5,B0853X2F4M,CELLULAR_PHONE_CASE,9.99,True
6,B07R91S92W,CELLULAR_PHONE_CASE,9.99,True
7,B0896LJNLH,DRINKING_CUP,81.99,True
8,B0853WVZ6Q,CELLULAR_PHONE_CASE,24.99,True
9,B07R51S57D,FURNITURE_COVER,129.99,True


#Sanity-check the prices

In [9]:
print(df['price'].describe())

med = df.groupby('ptype')['price'].median().sort_values()
print('Cheapest types by median price:')
print(med.head(10))
print('Most expensive types by median price:')
print(med.tail(10))

under50 = df[df['price'] < 50]
print(len(under50), 'items under $50')
print(under50['ptype'].value_counts().head(10))

# Determinism check: running the generator again must give identical prices
again = [synthetic_price(i, t) for i, t in zip(df['item_id'].head(1000), df['ptype'].head(1000))]
assert again == df['price'].head(1000).tolist(), 'Prices are not deterministic'
print('Determinism check passed')

count    147702.000000
mean         53.532599
std         106.129938
min           1.990000
25%          13.990000
50%          25.990000
75%          51.990000
max        2345.990000
Name: price, dtype: float64
Cheapest types by median price:
ptype
AIR_PURIFIER                    10.99
VEHICLE_SAFETY_CAMERA           11.99
CARRYING_CASE_OR_BAG            12.99
SUITCASE                        12.99
CELLULAR_PHONE_CASE             13.99
KICK_SCOOTER                    13.99
CE_CARRYING_CASE_OR_BAG         13.99
MEMORY_READER                   13.99
CAMERA_BAGS_AND_CASES           14.49
AMAZON_BOOK_READER_ACCESSORY    14.99
Name: price, dtype: float64
Most expensive types by median price:
ptype
PORTABLE_ELECTRONIC_DEVICE_COVER    185.99
DESK                                204.99
PORTABLE_ELECTRONIC_DEVICE_MOUNT    207.49
PORTABLE_TOOL_BOX                   207.99
PORTABLE_AUDIO                      217.99
SPORT_TABLE_GAME                    293.99
HOME_BED_AND_BATH                   397.

In [10]:
out = f'{PROJECT_DIR}/data/listings_en_priced.parquet'
df.to_parquet(out)
print('Saved', out, df.shape)

Saved /content/drive/MyDrive/IK classes/30_week 04Oct2026-LLM Ops - AI System Design- Sayan/ShopTalk/data/listings_en_priced.parquet (147702, 13)
